In [6]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/tam_pr/tcg_bot")
CONFIGS = ROOT / "configs"
PROC = ROOT / "data" / "processed"

cards = pd.read_parquet(PROC / "cards.parquet")
plays = pd.read_csv(CONFIGS / "plays.csv", encoding="utf-8")
constants = pd.read_csv(CONFIGS / "game_constants.csv", encoding="utf-8")
conditions = pd.read_csv(CONFIGS / "special_conditions.csv", encoding="utf-8")

In [7]:
plays["limit_per_turn"] = plays["limit_per_turn"].astype("Int64")        # empty = no limit
conditions["checkup_order"] = conditions["checkup_order"].astype("Int64")  # empty = not checked at Checkup
constants["value"] = constants["value"].astype("Int64")
for df in (plays, constants, conditions):
    df["source_page"] = df["source_page"].astype("string")                 # "8;18" is text, so keep every page column text

In [8]:
STATUSES = {"verified", "needs_check", "inferred"}
GROUPS = {"setup", "turn", "main", "attack", "checkup", "end"}

def check_columns(df, required, name):
    missing = set(required) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"

check_columns(plays, ["play_id", "group", "name", "card_kind", "limit_per_turn", "requires",
                      "effect_summary", "card_can_override", "source_page", "status"], "plays")
check_columns(constants, ["key", "value", "unit", "notes", "source_page", "status"], "game_constants")
check_columns(conditions, ["condition", "checkup_order", "checkup_effect", "blocks_attack", "blocks_retreat",
                           "exclusive_group", "removed_by", "source_page", "status"], "special_conditions")

for df, key, name in [(plays, "play_id", "plays"), (constants, "key", "game_constants"),
                      (conditions, "condition", "special_conditions")]:
    assert df[key].is_unique, f"{name}: duplicate {key}: {df.loc[df[key].duplicated(), key].tolist()}"
    assert df[key].notna().all(), f"{name}: empty {key}"
    bad = set(df["status"]) - STATUSES
    assert not bad, f"{name}: unknown status {bad}"
    assert df["source_page"].notna().all(), f"{name}: every row needs a source_page"

assert set(plays["group"]) <= GROUPS, set(plays["group"]) - GROUPS

# card kinds named in plays must exist in the cards table
known_kinds = set(cards["card_kind"].dropna())
named = {k for cell in plays["card_kind"].dropna() for k in cell.split(";")}
assert named <= known_kinds, f"unknown card_kind in plays: {named - known_kinds}"

# a constant may only be empty if it is still waiting to be checked
empty = constants[constants["value"].isna()]
assert (empty["status"] == "needs_check").all(), empty

# Checkup order must be 1..n with no gaps
order = conditions["checkup_order"].dropna().astype(int).sort_values().tolist()
assert order == list(range(1, len(order) + 1)), order
assert conditions["blocks_attack"].dtype == bool and conditions["blocks_retreat"].dtype == bool

In [9]:
open_items = pd.concat([
    plays.loc[plays["status"] != "verified", ["play_id", "status", "effect_summary"]].rename(columns={"play_id": "item"}).assign(table="plays"),
    constants.loc[constants["status"] != "verified", ["key", "status", "notes"]].rename(columns={"key": "item", "notes": "effect_summary"}).assign(table="game_constants"),
    conditions.loc[conditions["status"] != "verified", ["condition", "status", "checkup_effect"]].rename(columns={"condition": "item", "checkup_effect": "effect_summary"}).assign(table="special_conditions"),
], ignore_index=True)
print(f"{len(plays)} plays, {len(constants)} constants, {len(conditions)} special conditions")
print(len(open_items), "items still need checking:")
open_items[["table", "item", "status"]]


23 plays, 16 constants, 5 special conditions
4 items still need checking:


,table,item,status
0,plays,main_mega_turn_end,needs_check
1,plays,turn_end,inferred
2,game_constants,mega_ex_prize_value,needs_check
3,game_constants,resistance_amount,needs_check


Can end turn if player chooses to <br>
Turn does not end if it gets to a mega evolution under 2026 rules (still ask) <br>
Price value of a mega card under new rules, current 2 price cards

In [10]:
plays.to_parquet(PROC / "plays.parquet", index=False)
constants.to_parquet(PROC / "game_constants.parquet", index=False)
conditions.to_parquet(PROC / "special_conditions.parquet", index=False)